# Módulo 3 — 3/3 - Programación de tareas: cron, Programador de tareas y schedule

En esta clase vamos a cubrir:

1. Automatización basada en el tiempo
2. Tareas programadas en Windows: el Programador de tareas
3. `cron` en Linux y macOS
4. El módulo `schedule` para programar desde Python
5. Cómo elegir el método adecuado
6. Actividad práctica: resultados deportivos + notificación diaria
7. Un quiz interactivo y el examen del módulo

> La idea queresume la clase: en vez de que vos esperes a que los datos estén listos, el
> script corre a una hora fija mientras dormís. El ordenador queda como un asistente que
> trabaja 24/7.

Comprobamos el entorno.

In [1]:
import sys
import importlib

print("Python:", sys.version.split()[0])
print()

for paquete in ("schedule", "requests", "plyer"):
    try:
        importlib.import_module(paquete)
        print(f"{paquete:10} OK")
    except ImportError:
        print(f"{paquete:10} FALTA  ->  pip install {paquete}")

Python: 3.13.3

schedule   OK
requests   OK
plyer      OK


---

## 1. Automatización basada en el tiempo

Una tarea programada es un script que el sistema ejecuta **en un momento concreto o cada
cierto intervalo**, sin que nadie lo lance.

El caso motivador del módulo es muy real: hay un conjunto de datos enorme que hay que bajar
toda la noche. Programado a las 3 AM, a las 8 tenés los datos listos.

Lo que aporta:

| Capacidad | Qué significa en la práctica |
|-----------|------------------------------|
| Momento específico | "Esto pasa a las 23:30" |
| Intervalo periódico | "Esto pasa cada 5 minutos" |
| Varias veces al día | "Esto pasa a las 8:00, 13:00 y 18:00" |
| Gestión de errores | Si falla, queda registrado y no te enteras por sorpresa |

Hay tres caminos posibles, y la clase recorre los tres:

- **Nivel sistema**: `cron` (Linux/macOS) o el Programador de tareas (Windows)
- **Nivel Python**: el módulo `schedule`
- **Nivel nube**: Azure, AWS Lambda (mencionado como alternativa cuando la máquina puede
  estar apagada)

---

## 2. El Programador de tareas de Windows

El **Programador de tareas** es la herramienta nativa de Windows. Es el equivalente gráfico
de `cron`.

Motivos para usarlo:

- **Automatización**: configúralo una vez y olvídalo
- **Ahorro de tiempo**: los informes de la mañana se generan de noche
- **Fiabilidad**: la tarea corre aunque cierres sesión o reinicies

Limitaciones, y conviene conocerlas antes de confiarle tu vida:

- El equipo **debe estar encendido** en el momento de la ejecución
- Algunas tareas consumen muchos recursos e interfieren con el uso normal
- Conviene revisarlas periódicamente

Tareas típicas: limpieza de disco, actualización de software, copias de seguridad, análisis
de virus, mantenimiento del sistema.

### Cómo se crea una tarea

El asistente te pide, en este orden:

1. **Nombre** descriptivo (es lo primero que vas a ver al volver dentro de seis meses)
2. **Desencadenador**: diario, semanal, mensual o **por evento del sistema**
3. **Fecha y hora** de inicio
4. **Acción**: ejecutar un programa
5. **Ruta del script** de Python y sus argumentos
6. Revisión y listo

Para editar una tarea existente: botón derecho → *Propiedades*, y ahí las pestañas
**Activadores**, **Condiciones**, **Acciones** y **Configuración** (esta última incluye
*ejecutar con privilegios elevados*).

| Pestaña | Qué se configura |
|---------|------------------|
| **Activadores** | Cuándo se ejecuta: horarios, eventos, inicios de sesión |
| **Condiciones** | Solo si el equipo está inactivo, solo si hay red... |
| **Acciones** | Qué programa se ejecuta, con qué argumentos |
| **Configuración** | Tiempo máximo, reintentos, reiniciar si falla, privilegios |

### El script que va a ejecutar

Este es el "Hola mundo" de la programación de tareas: un script que imprime la hora en que
se ejecutó. Guardalo como `tarea_programada.py` en la carpeta que quieras.

In [2]:
# Contenido de tarea_programada.py
# El Programador de tareas lo invocaria como:
#     python.exe "C:\\ruta\\tarea_programada.py"
from datetime import datetime

print(f"[{datetime.now():%Y-%m-%d %H:%M:%S}] Tarea programada ejecutada")

[2026-10-26 20:25:12] Tarea programada ejecutada


In [ ]:
# Desde el notebook vemos las dos cosas que hay que copiar al Programador de tareas:
# 1) la ruta EXACTA del interprete de Python, y 2) como leer argumentos.
import sys

print("Interprete (va en 'Programa o script'):")
print("   ", sys.executable)

# 2) Los argumentos van en la pestana 'Agregar argumentos', separados por espacios.
#    Ejemplo:   C:\datos\salidas\  origen.csv  destino.xlsx
ARGUMENTOS_EJEMPLO = [r"C:\datos\salidas", "origen.csv", "destino.xlsx"]

def analizar_argumentos(argv):
    # El primero es siempre el nombre del propio script
    programa, *resto = argv
    print("  programa :", programa)
    for indice, valor in enumerate(resto):
        print(f"  argv[{indice}] :", valor)
    if resto:
        print("  -> carpeta :", resto[0])
        print("  -> entrada :", resto[1] if len(resto) > 1 else "(ninguna)")
        print("  -> salida  :", resto[2] if len(resto) > 2 else "(ninguna)")

analizar_argumentos(["tarea_programada.py"] + ARGUMENTOS_EJEMPLO)

In [3]:
# La recomendacion mas importante del modulo: RUTA ABSOLUTA siempre.
# El servicio arranca en una carpeta distinta de la tuya, asi que una ruta
# relativa como "datos/salidas" va a buscar en el sitio equivocado.
import os

# MAL: ruta relativa, dependera del directorio de trabajo del servicio
ruta_mala = "datos/salidas/informe.txt"

# BIEN: ruta absoluta
ruta_buena = os.path.abspath(ruta_mala)

print("  relativa  :", ruta_mala, "  <- se rompe si cambia el directorio")
print("  absoluta  :", ruta_buena)
print("  existe?   :", os.path.exists(ruta_buena))
print()
print("  Ademas: el script debe registrar su propia ejecucion con logging, para que")
print("  sepas que corio aunque nadie lo haya pedido.")

  relativa  : datos/salidas/informe.txt   <- se rompe si cambia el directorio
  absoluta  : d:\My Things\Curso-Py\Automatizacion\Modulo_3\datos\salidas\informe.txt
  existe?   : False

  Ademas: el script debe registrar su propia ejecucion con logging, para que
  sepas que corio aunque nadie lo haya pedido.


---

## 3. cron en Linux y macOS

`cron` es el planificador nativo de Unix. La configuración se escribe en el `crontab`, y
cada línea tiene **cinco campos de tiempo** seguidos del comando:

```
┌───── minuto (0-59)
│ ┌─── hora (0-23)
│ │ ┌─ día del mes (1-31)
│ │ │ ┌─ mes (1-12, o Ene-Dic)
│ │ │ │ ┌─ día de la semana (0-6, 0 = domingo)
│ │ │ │ │
* * * * *  comando
```

Un ejemplo de cada campo:

| Campo | Significado | Ejemplo | Se ejecuta |
|-------|-------------|---------|------------|
| minuto | 0-59 | `30 10 * * *` | a las 10:30, todos los días |
| hora | 0-23 (0 = medianoche) | `0 2 * * *` | a las 2:00, todos los días |
| día del mes | 1-31 | `0 0 15 * *` | a medianoche, los días 15 |
| mes | 1-12 o abreviado | `0 0 1 1 *` | el 1 de enero |
| día de la semana | 0-6 (0 = domingo) | `0 12 * * 5` | los viernes a las 12:00 |

Y los cuatro caracteres especiales, que son la mitad de la potencia de cron:

| Carácter | Significado | Ejemplo | Significado del ejemplo |
|----------|-------------|---------|-------------------------|
| `*` | todos los valores | `*/5 * * * *` | cada 5 minutos |
| `,` | lista de valores | `0,15,30,45 * * * *` | cada 15 minutos |
| `-` | rango | `0 0 * * 1-5` | de lunes a viernes |
| `/` | incremento | `0 */4 * * *` | cada 4 horas |

Ejemplos reales:

```bash
# Revision del sistema cada 5 minutos
*/5 * * * * /usr/local/bin/system_check.sh

# Informe de ventas a las 8:00 los dias 1 y 15 de cada mes
0 8 1,15 * * /usr/local/bin/generate_sales_report.py

# Reiniciar un servicio de lunes a viernes a medianoche
0 0 * * 1-5 /usr/local/bin/restart_service.sh

# Mantenimiento de base de datos cada 4 horas
0 */4 * * * /usr/local/bin/database_maintenance.sql
```

### Leer una expresión cron

Leer una línea de crontab de un vistazo es más difícil de lo que parece. Escribimos un
parser que la convierta a lenguaje natural: es la mejor forma de entenderlo de verdad y
evita tener que contar campos a mano.

In [ ]:
DIAS_SEMANA = {
    0: "domingo", 1: "lunes", 2: "martes", 3: "miercoles",
    4: "jueves", 5: "viernes", 6: "sabado",
}
MESES = {
    1: "enero", 2: "febrero", 3: "marzo", 4: "abril",
    5: "mayo", 6: "junio", 7: "julio", 8: "agosto",
    9: "septiembre", 10: "octubre", 11: "noviembre", 12: "diciembre",
}

RANGOS = {"minuto": (0, 59), "hora": (0, 23), "dia_mes": (1, 31),
          "mes": (1, 12), "dia_semana": (0, 6)}

NOMBRES_DIA = {v: k for k, v in DIAS_SEMANA.items()}
NOMBRES_MES = {v.capitalize(): k for k, v in MESES.items()}


def describir_campo(valor, tipo):
    # Resuelve los nombres que cron acepta en texto
    valor = valor.strip()
    if tipo == "dia_semana":
        for nombre, numero in NOMBRES_DIA.items():
            if valor.lower().startswith(nombre[:3]):
                valor = str(numero)
    elif tipo == "mes":
        for nombre, numero in NOMBRES_MES.items():
            if valor.lower().startswith(nombre[:3].lower()):
                valor = str(numero)

    if valor == "*":
        return f"cada {tipo.replace('_', ' ')}"
    if valor.startswith("*/"):
        paso = valor[2:]
        unidad = {"minuto": "minutos", "hora": "horas", "dia_mes": "dias",
                  "mes": "meses", "dia_semana": "dias"}[tipo]
        return f"cada {paso} {unidad}"
    if "," in valor:
        partes = []
        for trozo in valor.split(","):
            numero = int(trozo)
            if tipo == "dia_semana":
                partes.append(DIAS_SEMANA.get(numero, trozo))
            elif tipo == "mes":
                partes.append(MESES.get(numero, trozo))
            else:
                partes.append(trozo)
        return "a las " + ", ".join(partes) if tipo in ("minuto", "hora") else "los " + ", ".join(partes)
    if "-" in valor:
        inicio, fin = valor.split("-")
        if tipo == "dia_semana":
            return f"de {DIAS_SEMANA.get(int(inicio))} a {DIAS_SEMANA.get(int(fin))}"
        return f"del {inicio} al {fin}"
    return f"el valor {valor}"


def describir_cron(expresion):
    partes = expresion.split()
    if len(partes) < 6:
        return f"Expresion incompleta: {expresion}"

    minuto, hora, dia_mes, mes, dia_semana, *resto = partes
    comando = " ".join(resto)

    when = []
    if minuto != "*":
        when.append(describir_campo(minuto, "minuto"))
    if hora != "*":
        if hora.startswith("*/"):
            when.append(f"cada {hora[2:]} horas")
        else:
            hora_num = int(hora.split(",")[0].split("-")[0])
            if minuto == "*":
                when.append(f"a las {hora_num:02d}:00")
            elif minuto.isdigit():
                when.append(f"a las {hora_num:02d}:{minuto.zfill(2)}")
            else:
                when.append(f"a las {hora_num:02d}:00 ({minuto})")
    if dia_mes != "*":
        when.append(describir_campo(dia_mes, "dia_mes"))
    if mes != "*":
        when.append(f"en {describir_campo(mes, 'mes')}")
    if dia_semana != "*":
        when.append(f"los {describir_campo(dia_semana, 'dia_semana')}")

    momento = " ".join(when) if when else "minutamente"
    return f"{momento}  ->  {comando}"

In [ ]:
EXPRESIONES = [
    "30 10 * * * /usr/local/bin/informe.sh",
    "*/5 * * * * /usr/local/bin/check.sh",
    "0 0 * * 1-5 /usr/local/bin/rotar_logs.sh",
    "0 8 1,15 * * /usr/local/bin/ventas.py",
    "0 */4 * * * /usr/local/bin/mantenimiento.sql",
    "0 5 * * 0 /usr/bin/db_backup.sh",
]

for expresion in EXPRESIONES:
    print(describir_cron(expresion))

### Validar antes de instalar

Un error de escritura en el crontab no avisa: simplemente la tarea no corre. Un validador
sencillo es la mejor inversión de tiempo del módulo.

In [ ]:
def validar_cron(expresion):
    partes = expresion.split()
    if len(partes) < 6:
        return False, "Faltan campos: se esperan 5 campos de tiempo mas el comando"

    for campo, tipo in zip(partes[:5], ("minuto", "hora", "dia_mes", "mes", "dia_semana")):
        minimo, maximo = RANGOS[tipo]
        for trozo in campo.split(","):
            if trozo == "*":
                continue
            base = trozo.split("/")[0]
            for item in base.split("-"):
                if item.isdigit():
                    if not (minimo <= int(item) <= maximo):
                        return False, f"{tipo}: {item} fuera del rango {minimo}-{maximo}"
                elif tipo in ("mes", "dia_semana"):
                    pass   # nombre abreviado (Ene, Lun): se acepta
                else:
                    return False, f"{tipo}: {item!r} no es un numero valido"
    return True, "Expresion correcta"

PRUEBAS = [
    "30 10 * * * /usr/local/bin/informe.sh",
    "0 5 * * 0 /usr/bin/db_backup.sh",
    "99 10 * * * /bin/roto",          # minuto invalido
    "30 25 * * * /bin/roto",          # hora invalida
    "0 0 32 * * /bin/roto",           # dia de mes inexistente
    "* * * *",                        # incompleta
]

for expresion in PRUEBAS:
    correcta, motivo = validar_cron(expresion)
    marca = "OK  " if correcta else "MAL "
    print(f"{marca} {expresion:45} {motivo}")

### El reto de cron

**Copias de seguridad a las 11:30 PM, todos los días**, del script `/usr/bin/backup_database.sh`.

Los cinco campos van en este orden: minuto, hora, día del mes, mes, día de la semana. A las
11:30 PM son `30 23`, y "todos los días" es `*` en los tres campos de fecha.

In [ ]:
# El reto, resuelto
cron_job_input = "30 23 * * * /usr/bin/backup_database.sh"

print("Tu respuesta:")
print("  ", cron_job_input)
print()
print("En palabras:", describir_cron(cron_job_input))

# Un avisao sobre las dos confusiones mas comunes
print()
print("Recordatorio:")
print('  "a las 11:30 PM" es 30 23, no 11 23  <- el campo hora va en formato 24 h')
print('  "todos los dias" es * * *             <- no 0-6 en el dia de la semana')

---

## 4. El módulo schedule

`schedule` es la opción **dentro de Python**: tú escribes la función (qué) y el horario
(cuándo), y el módulo se encarga del resto.

Ventajas:

- Ligero, sin configuración ni software adicional
- **Multiplataforma**: Windows, macOS y Linux
- La lógica se queda en tu código, no en un archivo de sistema

Reglas de la sintaxis:

```python
schedule.every(5).minutes.do(tarea)      # cada 5 minutos
schedule.every(30).seconds.do(tarea)     # cada 30 segundos
schedule.every().saturday.do(tarea)      # todos los sábados
schedule.every().saturday.at("23:30").do(tarea)   # los sábados a las 23:30
schedule.every().day.at("08:00").do(tarea).tag("recordatorios")   # con etiqueta
```

`schedule` registra las tareas, pero no crea un proceso que quede trabajando solo. Alguien
tiene que llamar a `run_pending()` repetidamente mientras el programa siga activo. La forma
directa es un bucle en el hilo principal:

```python
while True:
    schedule.run_pending()
    time.sleep(1)
```

Si el programa también tiene que hacer otro trabajo, se puede ejecutar ese bucle en un
**hilo de fondo**. Así el hilo principal continúa con la aplicación mientras el hilo del
planificador revisa los horarios. El hilo no es un proceso separado: si el programa termina
o el equipo se apaga, las tareas dejan de ejecutarse. `schedule` tampoco conserva las tareas
después de reiniciar el programa.

In [4]:
import time
import schedule
from datetime import datetime

def registrar(tarea):
    print(f"[{datetime.now():%H:%M:%S}] {tarea}")

# --- Intervalos ---
schedule.every(2).seconds.do(registrar, "chequeo rapido")
schedule.every(5).minutes.do(registrar, "revisar logs")
schedule.every(30).seconds.do(registrar, "latido")

# --- Dias de la semana (encadenables) ---
schedule.every().saturday.do(registrar, "pagar nomina")

# --- Momento exacto, en formato 24 h ---
schedule.every().day.at("08:00").do(registrar, "informe diario")
schedule.every().saturday.at("23:30").do(registrar, "backup semanal")

# --- Etiquetas para agrupar ---
schedule.every().sunday.do(registrar, "desfragmentar disco").tag("mantenimiento")
schedule.every().day.at("00:00").do(registrar, "backup nocturno").tag("mantenimiento")

# --- Inspeccion ---
print("Tareas programadas:")
for trabajo in schedule.get_jobs():
    proximo = trabajo.next_run.strftime("%Y-%m-%d %H:%M:%S") if trabajo.next_run else "?"
    print(f"  {str(trabajo.at_time):8} cada {str(trabajo.interval):14} -> {trabajo.job_func.__name__:18} [{','.join(trabajo.tags) or 'sin etiqueta'}]")
    print(f"             proxima: {proximo}")

print()
print("Solo las de mantenimiento:")
for trabajo in schedule.get_jobs(tag="mantenimiento"):
    print("  ", trabajo.job_func.__name__)

Tareas programadas:
  None     cada 2              -> registrar          [sin etiqueta]
             proxima: 2026-10-26 20:34:13
  None     cada 5              -> registrar          [sin etiqueta]
             proxima: 2026-10-26 20:39:11
  None     cada 30             -> registrar          [sin etiqueta]
             proxima: 2026-10-26 20:34:41
  None     cada 1              -> registrar          [sin etiqueta]
             proxima: 2026-10-31 20:34:11
  08:00:00 cada 1              -> registrar          [sin etiqueta]
             proxima: 2026-10-27 08:00:00
  23:30:00 cada 1              -> registrar          [sin etiqueta]
             proxima: 2026-10-31 23:30:00
  None     cada 1              -> registrar          [mantenimiento]
             proxima: 2026-11-01 20:34:11
  00:00:00 cada 1              -> registrar          [mantenimiento]
             proxima: 2026-10-27 00:00:00

Solo las de mantenimiento:
   registrar
   registrar


In [ ]:
# El bucle real es infinito. Aqui lo limitamos a 5 vueltas para poder ejecutarlo.
# Para el bucle de verdad, quitale el limite:
#     while True:
#         schedule.run_pending()
#         time.sleep(1)

def run_scheduler(vueltas=5):
    for _ in range(vueltas):
        schedule.run_pending()
        time.sleep(1)

print("Ejecutando el planificador 5 vueltas...")
run_scheduler(vueltas=5)
print("\nSolo se disparo lo que ya habia llegado a su hora (los intervalos de segundos).")

In [ ]:
# Para reutilizar el kernel sin reiniciarlo, se limpian las tareas
print("Tareas antes de limpiar:", len(schedule.get_jobs()))
schedule.clear()
print("Tareas despues de limpiar:", len(schedule.get_jobs()), "\n")

### Ejemplo: Pomodoro Lite con un hilo de fondo

Este ejemplo acortado alterna **10 segundos de trabajo y 5 segundos de descanso** para que
puedas ver el ciclo enseguida. El hilo principal representa el trabajo normal de la aplicación;
el hilo secundario ejecuta `schedule.run_pending()` y muestra los avisos con Plyer.

Ejecuta esta celda como un programa que queda activo. Para detenerlo, interrumpe la celda
(en VS Code/Jupyter, botón **Interrupt** o el botón de detener). Al detener el programa también
se detiene el planificador. Para usar tiempos Pomodoro habituales, cambia las constantes a
25 minutos y 5 minutos.

In [3]:
import threading
import time

import schedule
from plyer import notification

TRABAJO_SEGUNDOS = 10
DESCANSO_SEGUNDOS = 5
detener_planificador = threading.Event()


def avisar(titulo, mensaje):
    print(mensaje)
    notification.notify(title=titulo, message=mensaje, timeout=5)


def comenzar_descanso():
    avisar("Pomodoro Lite", "Terminó el bloque de trabajo: hora de descansar.")
    schedule.every(DESCANSO_SEGUNDOS).seconds.do(volver_al_trabajo).tag("pomodoro")
    return schedule.CancelJob


def volver_al_trabajo():
    avisar("Pomodoro Lite", "Terminó el descanso: volvemos al trabajo.")
    schedule.every(TRABAJO_SEGUNDOS).seconds.do(comenzar_descanso).tag("pomodoro")
    return schedule.CancelJob


def revisar_horarios():
    while not detener_planificador.is_set():
        schedule.run_pending()
        detener_planificador.wait(0.5)


# Quitamos solo trabajos anteriores de este ejemplo, no otros trabajos schedule.
schedule.clear("pomodoro")
schedule.every(TRABAJO_SEGUNDOS).seconds.do(comenzar_descanso).tag("pomodoro")
hilo_planificador = threading.Thread(target=revisar_horarios, daemon=True)
hilo_planificador.start()

try:
    while True:
        print("El programa principal sigue trabajando...")
        time.sleep(3)
except KeyboardInterrupt:
    print("\nDeteniendo Pomodoro Lite...")
finally:
    detener_planificador.set()
    hilo_planificador.join()
    schedule.clear("pomodoro")
    print("Planificador detenido.")

El programa principal sigue trabajando...
El programa principal sigue trabajando...
El programa principal sigue trabajando...
El programa principal sigue trabajando...
Terminó el bloque de trabajo: hora de descansar.
El programa principal sigue trabajando...
El programa principal sigue trabajando...
Terminó el descanso: volvemos al trabajo.
El programa principal sigue trabajando...
El programa principal sigue trabajando...
El programa principal sigue trabajando...
Terminó el bloque de trabajo: hora de descansar.
El programa principal sigue trabajando...
El programa principal sigue trabajando...
Terminó el descanso: volvemos al trabajo.
El programa principal sigue trabajando...
El programa principal sigue trabajando...
El programa principal sigue trabajando...
Terminó el bloque de trabajo: hora de descansar.

Deteniendo Pomodoro Lite...
Planificador detenido.


### El reto de `schedule`: un sistema de recordatorios

El ejercicio pide cuatro tareas sobre un equipo EfficientOps:

| Tarea | Horario |
|-------|---------|
| `send_daily_reminder` | todos los días a las 10:00, etiqueta `reminders` |
| `send_weekly_reminder` | los lunes a las 9:00, etiqueta `reminders` |
| `calculate_volunteer_hours` | los martes a las 23:59 |
| `calculate_volunteer_hours` | los viernes a las 23:59 |

Resuelto:

In [ ]:
def send_daily_reminder():
    print("Don't forget to update the project status in the tracking tool!")

def send_weekly_reminder():
    print("Reminder: Weekly team meeting today at 9:00 AM!")

def calculate_volunteer_hours():
    print("Calculating volunteer hours. Results will be emailed!")

# Diarios a las 10 AM, con etiqueta
schedule.every().day.at("10:00").do(send_daily_reminder).tag("reminders")

# Los lunes a las 9 AM, con etiqueta
schedule.every().monday.at("09:00").do(send_weekly_reminder).tag("reminders")

# Martes y viernes a las 23:59
schedule.every().tuesday.at("23:59").do(calculate_volunteer_hours)
schedule.every().friday.at("23:59").do(calculate_volunteer_hours)

print("Listas para el equipo EfficientOps:")
for trabajo in schedule.get_jobs():
    dia = trabajo.start_day if hasattr(trabajo, "start_day") and trabajo.start_day else "todos los dias"
    print(f"  {trabajo.at_time:6} {dia:14} -> {trabajo.job_func.__name__:26} [{','.join(trabajo.tags) or '-'}]")

print("\nSolo las de recordatorios:")
for trabajo in schedule.get_jobs(tag="reminders"):
    print(f"  {trabajo.at_time} {trabajo.job_func.__name__}")

schedule.clear()

---

## 5. Cómo elegir el método de programación

La pregunta del módulo es práctica: cuando te piden automatizar algo, ¿qué usás?

Los cinco criterios de selección:

1. **Compatibilidad del sistema**: ¿Unix, Windows o ambos?
2. **Complejidad**: ¿una recurrencia simple o dependencias y desencadenadores condicionales?
3. **Facilidad de uso**: ¿qué curva de aprendizaje tiene tu equipo?
4. **Necesidades de integración**: ¿necesitas eventos del sistema o procesos externos?
5. **Familiaridad del equipo**: lo que ya conocen suele ser más eficiente

| Método | Sistema | Lógica en tu código | Curva | Para qué |
|--------|---------|---------------------|-------|----------|
| **`schedule`** | Multiplataforma | **Sí** | Mínima | Recurrencias simples, eventos únicos |
| **Programador de tareas** | Windows | No | Media | Eventos del sistema, inicios de sesión, GUI |
| **`cron`** | Unix/macOS | No | Media | Backups, rotación de logs, del SO |

Un cuarto nombre aparece cuando la cosa crece: **APScheduler** para persistencia y control de
concurrencia, y **Celery** para colas de trabajo distribuidas.

In [ ]:
def recomendar_metodo(sistema="windows", complejidad="simple",
                       necesita_eventos_sistema=False, familiaridad_equipo=""):
    # Devuelve (metodo, motivo)
    if sistema == "linux" or sistema == "mac":
        if complejidad == "compleja" or necesita_eventos_sistema:
            return "cron", "Unix puro: el planificador nativo y fiable del sistema."
        return "cron", "Servidor Unix: cron es la opcion estandar y no depende de Python."

    if sistema == "windows":
        if complejidad == "compleja" or necesita_eventos_sistema:
            return "Programador de tareas", "Windows con desencadenadores por evento del sistema."
        return "Programador de tareas", "Windows: la herramienta nativa, sin depender de Python."

    # multiplataforma
    if complejidad == "compleja":
        return "APScheduler", "Multiplataforma pero complejo: hace falta persistencia y control de concurrencia."
    return "schedule", "Multiplataforma y logica dentro del propio codigo Python."

ESCENARIOS = [
    dict(sistema="linux",    complejidad="simple", familiaridad_equipo=""),
    dict(sistema="windows",  complejidad="compleja", necesita_eventos_sistema=True, familiaridad_equipo=""),
    dict(sistema="linux",    complejidad="compleja", familiaridad_equipo=""),
    dict(sistema="multi",    complejidad="simple", familiaridad_equipo=""),
    dict(sistema="multi",    complejidad="compleja", familiaridad_equipo=""),
]

for escenario in ESCENARIOS:
    metodo, motivo = recomendar_metodo(**escenario)
    sistema = escenario["sistema"].ljust(8)
    print(f"{sistema} {escenario['complejidad']:9} -> {metodo}")
    print(f"         {motivo}")

### Cuándo el script se queda corto

El módulo es explícito: si una tarea que corre cada 10 minutos **consume muchos recursos y
choca con otros procesos**, `schedule` no es la respuesta. Porque `schedule` no prioriza, no
limita recursos ni distribuye carga: solo dispara.

Ahí el salto es a **Celery** (colas y workers) o **APScheduler** (persistencia y concurrencia).
En la nube, a **AWS Lambda** o **Google Cloud Functions**.

---

## 6. Actividad práctica: resultados deportivos y notificación diaria

El escenario de la actividad integradora: sos analista de datos en un sitio web de
noticias deportivas. Hay que mantener actualizado un conjunto de datos y que el equipo lo
reciba cada mañana sin pedirlo.

Las cuatro partes:

| Parte | Qué se hace |
|-------|-------------|
| 1 | Importar `requests`, `plyer`, `schedule` y `time` |
| 2 | `fetch_scores()` contra la API de football-data.org |
| 3 | `send_notification()` con `plyer.notification` |
| 4 | Programar con `schedule` a las 08:00 y lanzar el bucle |

La API elegida es **https://www.football-data.org/**, que da una clave gratuita al
registrarse. La clave va en la cabecera `X-Auth-Token`.

### Parte 1: las librerías

```bash
pip install requests plyer schedule
```

In [ ]:
import os
import time
from datetime import datetime

import requests
from dotenv import load_dotenv

load_dotenv()

print("Python:", __import__("sys").version.split()[0])

# plyer es opcional: solo se usa para la notificacion de escritorio
try:
    from plyer import notification
    HAY_PLYER = True
except ImportError:
    HAY_PLYER = False

print("requests    OK")
print("schedule    OK")
print("plyer      ", "OK" if HAY_PLYER else "no instalado (se mostrara la version simulada)")

### Parte 2: obtener los resultados

Hay dos modos: sin `FOOTBALL_API_KEY` se usan datos simulados para practicar el flujo sin
conexión; con la clave configurada se consulta la API real. Si la consulta real falla, se
muestra el error en vez de reemplazarlo silenciosamente por datos inventados.

In [ ]:
API_FOOTBALL = "https://api.football-data.org/v4/matches"

# Datos simulados con el mismo esquema de marcador que devuelve football-data.org v4.
PARTIDOS_SIMULADOS = {
    "matches": [
        {
            "homeTeam": {"name": "River Plate"},
            "awayTeam": {"name": "Boca Juniors"},
            "score": {"fullTime": {"home": 2, "away": 1}},
            "status": "FINISHED",
        },
        {
            "homeTeam": {"name": "Racing Club"},
            "awayTeam": {"name": "Independiente"},
            "score": {"fullTime": {"home": 0, "away": 0}},
            "status": "FINISHED",
        },
    ]
}


def formatear_partido(partido):
    local = (partido.get("homeTeam") or {}).get("name", "Equipo local")
    visita = (partido.get("awayTeam") or {}).get("name", "Equipo visitante")
    marcador = ((partido.get("score") or {}).get("fullTime") or {})
    goles_local = marcador.get("home")
    goles_visita = marcador.get("away")

    if goles_local is None or goles_visita is None:
        return f"{local} vs {visita} ({partido.get('status', 'sin marcador')})"
    return f"{local} {goles_local} - {goles_visita} {visita}"


def fetch_scores(api_key=None):
    if not api_key:
        print("  [modo simulado] sin FOOTBALL_API_KEY: uso datos de ejemplo")
        return "\n".join(formatear_partido(p) for p in PARTIDOS_SIMULADOS["matches"])

    headers = {"X-Auth-Token": api_key}
    try:
        response = requests.get(API_FOOTBALL, headers=headers, timeout=15)
        response.raise_for_status()
        data = response.json()
    except (requests.exceptions.RequestException, ValueError) as e:
        return f"Error al consultar football-data.org: {e}"

    matches = data.get("matches") if isinstance(data, dict) else None
    if not isinstance(matches, list):
        return "Error: la respuesta de la API no contiene una lista 'matches'."
    if not matches:
        return "No hay partidos disponibles en este momento."
    return "\n".join(formatear_partido(p) for p in matches[:5])


CLAVE_FOOTBALL = os.getenv("FOOTBALL_API_KEY")
print("Resultado de fetch_scores():\n")
print(fetch_scores(CLAVE_FOOTBALL))

Para proteger la clave, no la escribas en la notebook. En la raíz del proyecto, copia
`.env.ejemplo` como `.env` y completa `FOOTBALL_API_KEY` allí. El archivo `.env` está
excluido de Git. `load_dotenv()` carga esa variable al iniciar la celda de librerías.

Después de crear o cambiar `.env`, reiniciá el kernel y volvé a ejecutar las celdas desde
el principio. También podés configurar una variable de entorno de Windows con `setx`, pero
tenés que reiniciar VS Code para que el kernel herede el nuevo valor.

### Parte 3: la notificación de escritorio

`plyer` es una capa sencilla para pedirle al sistema operativo que muestre una notificación.
No consulta la API ni programa cuándo ejecutarse: esas tareas corresponden a `requests` y
`schedule`, respectivamente. La llamada principal es:

```python
notification.notify(title="Título", message="Texto", timeout=10)
```

- `title`: encabezado de la notificación.
- `message`: texto que verá la persona.
- `timeout`: tiempo sugerido para mostrarla (el sistema puede ajustarlo).

La notificación aparece en el escritorio donde se está ejecutando Python. Si el kernel está
en un servidor remoto, contenedor o entorno sin escritorio, no necesariamente aparecerá en
la computadora desde la que abriste el notebook. También depende de los permisos y ajustes
de notificaciones del sistema.

> Dejamos `MOSTRAR_NOTIFICACION = False` para evitar que ejecutar el notebook te llene el
> escritorio de avisos. Para probar solo Plyer, después de ejecutar la celda siguiente,
> cambialo a `True` y llama `send_notification("Aviso de prueba")`; así no se consulta la API.

In [ ]:
MOSTRAR_NOTIFICACION = False   # ponelo en True para ver las notificaciones reales


def send_notification(mensaje=None):
    if mensaje is None:
        mensaje = fetch_scores(CLAVE_FOOTBALL)

    if HAY_PLYER and MOSTRAR_NOTIFICACION:
        notification.notify(
            title="Live Sports Score Update",
            message=mensaje,
            app_icon=None,
            timeout=10,
        )
        print("[notificacion enviada al escritorio]")
    else:
        # Sin plyer o con el interruptor apagado, mostramos lo que se habria enviado
        print("[notificacion simulada]")
        print(f"    titulo : Live Sports Score Update")
        print(f"    mensaje: {mensaje}")

    return mensaje


print()
send_notification()

### Parte 4: automatizar con `schedule`

In [ ]:
import schedule

# Todos los dias a las 08:00
schedule.every().day.at("08:00").do(send_notification)

print("Tareas programadas:")
for trabajo in schedule.get_jobs():
    proximo = trabajo.next_run.strftime("%Y-%m-%d %H:%M:%S") if trabajo.next_run else "?"
    print(f"  {trabajo.at_time} -> {trabajo.job_func.__name__}   proxima: {proximo}")

In [ ]:
# El bucle de la actividad es infinito:
#
#     while True:
#         schedule.run_pending()
#         time.sleep(1)
#
# En el notebook lo limitamos a 3 vueltas para poder ejecutarlo.
def run_scheduler(vueltas=3):
    for _ in range(vueltas):
        schedule.run_pending()
        time.sleep(1)

print("Probando el planificador (3 vueltas)...")
run_scheduler(vueltas=3)
print("\nLas 08:00 de hoy ya pasaron, asi que todavia no se dispara nada.")
print("Para probarlo de verdad, cambia la hora a un minuto o dos mas adelante:")
print('    schedule.clear()')
print('    schedule.every(1).minutes.do(send_notification)')

In [ ]:
# Una comprobacion util antes de dar por buena la automatizacion:
# que la funcion se pueda llamar a mano y devolver algo printable.
print("Llamada directa a send_notification():")
resultado = send_notification()
print()
print("Tipo de dato que devuelve:", type(resultado).__name__)
print("Sirve para el mensaje:", bool(resultado))

schedule.clear()

### Variación: notificar cada hora

La actividad pregunta cómo cambiar el horario. Es una línea:

In [ ]:
# En vez de a las 08:00, cada hora
schedule.every().hour.do(send_notification)

for trabajo in schedule.get_jobs():
    print(f"  cada {trabajo.interval} -> {trabajo.job_func.__name__}")
    print(f"  proxima ejecucion: {trabajo.next_run}")

# Ojo: .hour y .at() no se combinan. .at("08:00") es un momento concreto del dia;
# .every(1).hours es un intervalo. Son dos cosas distintas.

schedule.clear()

---

# Quiz interactivo: Programación de tareas

Responde cada pregunta y pulsa **Ver respuesta**. Hay preguntas de una sola opcion y de multiple respuesta (selecciona todas las que correspondan). Si fallas, lee la explicacion: ahi esta el porque.

In [ ]:
# --- Configuracion del quiz ---
# 1) Localizar la raiz del proyecto (la que contiene la carpeta 'quiz') y anadirla al sys.path
import sys
from pathlib import Path

RAIZ = Path.cwd()
while RAIZ.parent != RAIZ and not (RAIZ / "quiz").exists():
    RAIZ = RAIZ.parent
if str(RAIZ) not in sys.path:
    sys.path.append(str(RAIZ))

# 2) Importar funciones del modulo reutilizable
from quiz.quiz import aplicar_estilos, crear_quiz, crear_quiz_multiple

# 3) Aplicar el CSS (legible en VS Code y JupyterLab)
aplicar_estilos()

# 4) Cargar respuestas y explicaciones desde el JSON (misma carpeta que el notebook)
import json
with open(Path.cwd() / "Modulo_3_Programacion_Tareas_preguntas.json", encoding="utf-8") as fh:
    respuestas = json.load(fh)

print("Quiz listo:", ", ".join(respuestas.keys()))

### Pregunta 1

Has estado trabajando muchas horas y un colega te sugiere usar tareas programadas para racionalizar tu carga de trabajo. ¿Cuáles de las siguientes son ventajas de utilizar tareas programadas para la automatización basada en el tiempo?

- Permite ejecutar secuencias de comandos a horas o intervalos específicos
- Puede realizar varias tareas simultáneamente, lo que mejora la eficacia
- Reduce la necesidad de intervención manual y de trasnochar
- Garantiza la ejecución sin errores de las tareas

In [ ]:
q1_data = respuestas["q1"]
crear_quiz_multiple(q1_data["opciones"], q1_data["correctas"], q1_data["explicacion"])

### Pregunta 2

Tu ordenador funciona con lentitud y sospechas de la acumulación de archivos innecesarios y software obsoleto. ¿Qué tareas de mantenimiento se pueden automatizar mediante tareas programadas?

- Análisis de virus para comprobar y eliminar periódicamente el software malicioso.
- Actualizaciones de hardware para sustituir componentes obsoletos como RAM y discos duros.
- Actualizaciones de software para garantizar que dispones de las últimas versiones y parches de seguridad.
- Limpieza de disco para eliminar archivos temporales y liberar espacio de almacenamiento.

In [ ]:
q2_data = respuestas["q2"]
crear_quiz_multiple(q2_data["opciones"], q2_data["correctas"], q2_data["explicacion"])

### Pregunta 3

Necesitas configurar copias de seguridad automáticas para una base de datos crítica en tu servidor Linux, con un script llamado /usr/bin/db_backup.sh. Usando cron, ¿cómo programarías este script para que se ejecute a las 5:00 AM todos los domingos?

- 0 5 * * 0 /usr/bin/db_backup.sh
- * 5 0 0 0 /usr/bin/db_backup.sh
- 5 0 * * 7 /usr/bin/db_backup.sh
- 5 0 0 * * /usr/bin/db_backup.sh

In [ ]:
q3_data = respuestas["q3"]
crear_quiz(q3_data["opciones"], q3_data["correcta"], q3_data["explicacion"])

### Pregunta 4

Has escrito un script de Python que genera un informe diario de ventas y quieres automatizar su ejecución con el planificador de tareas. ¿Qué pasos están involucrados en configurar la tarea para ejecutar tu script Python?

- Especificar la acción de ejecutar un programa externo, que en este caso es el intérprete de Python ejecutando tu script
- Establecer un desencadenador para definir cuándo debe ejecutarse la tarea, por ejemplo, diariamente, semanalmente o en eventos específicos
- Dar un nombre descriptivo a la tarea para identificarla fácilmente más adelante
- Ejecutar manualmente el script cada día para asegurarse de que funciona correctamente

In [ ]:
q4_data = respuestas["q4"]
crear_quiz_multiple(q4_data["opciones"], q4_data["correctas"], q4_data["explicacion"])

### Pregunta 5

Estás trabajando en un proyecto que requiere obtener datos meteorológicos en tiempo real cada hora y has decidido utilizar el módulo schedule de Python. ¿Qué opción describe la función principal de este módulo en este caso?

- Se conecta directamente a las APIs meteorológicas y recupera los datos necesarios.
- Proporciona un marco para construir algoritmos complejos de análisis meteorológico.
- Genera visualizaciones e informes basados en los datos meteorológicos recopilados.
- Permite definir la hora y la frecuencia exactas de obtención de los datos meteorológicos.

In [ ]:
q5_data = respuestas["q5"]
crear_quiz(q5_data["opciones"], q5_data["correcta"], q5_data["explicacion"])

### Pregunta 6

Eres administrador de un servidor Linux y necesitas programar un script que realice copias de seguridad nocturnas de la base de datos a las 2:00 AM. ¿Qué método de programación sería el más adecuado?

- Programador de tareas
- Cron
- Ejecutar manualmente el script cada noche
- El módulo schedule

In [ ]:
q6_data = respuestas["q6"]
crear_quiz(q6_data["opciones"], q6_data["correcta"], q6_data["explicacion"])

## ¡Fin del quiz!

Si fallaste alguna, releé la explicación. La clave de la sección 4 es entender que
`schedule` **no** prioriza ni distribuye carga: cuando la tarea crece, el salto es a
APScheduler o Celery.

---

## Examen del módulo

Las 10 preguntas del examen final del módulo 3, tal como están en el material del curso.
Sirven como autoevaluación antes de cerrar la asignatura.

---

# Quiz interactivo: Examen del módulo 3

Responde cada pregunta y pulsa **Ver respuesta**. Hay preguntas de una sola opcion y de multiple respuesta (selecciona todas las que correspondan). Si fallas, lee la explicacion: ahi esta el porque.

In [ ]:
# --- Configuracion del quiz ---
# 1) Localizar la raiz del proyecto (la que contiene la carpeta 'quiz') y anadirla al sys.path
import sys
from pathlib import Path

RAIZ = Path.cwd()
while RAIZ.parent != RAIZ and not (RAIZ / "quiz").exists():
    RAIZ = RAIZ.parent
if str(RAIZ) not in sys.path:
    sys.path.append(str(RAIZ))

# 2) Importar funciones del modulo reutilizable
from quiz.quiz import aplicar_estilos, crear_quiz, crear_quiz_multiple

# 3) Aplicar el CSS (legible en VS Code y JupyterLab)
aplicar_estilos()

# 4) Cargar respuestas y explicaciones desde el JSON (misma carpeta que el notebook)
import json
with open(Path.cwd() / "Modulo_3_Examen_preguntas.json", encoding="utf-8") as fh:
    respuestas = json.load(fh)

print("Quiz listo:", ", ".join(respuestas.keys()))

### Pregunta 1

Estás diseñando una API para un sistema de gestión de libros de una biblioteca y necesitas actualizar los detalles de un libro existente, como su título o autor. ¿Qué método HTTP sería el más apropiado?

- POST
- PUT
- GET
- DELETE

In [ ]:
q1_data = respuestas["q1"]
crear_quiz(q1_data["opciones"], q1_data["correcta"], q1_data["explicacion"])

### Pregunta 2

Mientras trabajas con una API, recibes una respuesta con el código de estado 401. ¿Cuál es la razón más probable de este código de estado?

- El servidor está experimentando una sobrecarga temporal.
- El recurso solicitado no se ha podido encontrar en el servidor.
- La solicitud estaba malformada o le faltaban parámetros requeridos.
- La solicitud requiere credenciales de autenticación para acceder al recurso.

In [ ]:
q2_data = respuestas["q2"]
crear_quiz(q2_data["opciones"], q2_data["correcta"], q2_data["explicacion"])

### Pregunta 3

Tu aplicación encuentra un error 503 Service Unavailable mientras interactúa con una API. ¿Cuál es la causa más probable de este error?

- Su aplicación no ha podido autenticarse con el servidor API.
- Ha superado el límite de velocidad de la API.
- Faltan parámetros obligatorios en su solicitud de API.
- El servidor de la API no puede atender solicitudes temporalmente.

In [ ]:
q3_data = respuestas["q3"]
crear_quiz(q3_data["opciones"], q3_data["correcta"], q3_data["explicacion"])

### Pregunta 4

Como director de marketing, desea aumentar la eficacia y la personalización de sus campañas automatizando el envío de mensajes dirigidos a los clientes en función de su comportamiento y sus preferencias. ¿Qué enfoque lograría mejor este objetivo?

- Confíe únicamente en los algoritmos de las redes sociales para dirigir automáticamente los anuncios a su base de clientes deseada.
- Utilice Python para analizar los datos de los clientes y automatizar el envío de mensajes dirigidos a través de diversos canales.
- Contrate a un equipo de asistentes de marketing para realizar un seguimiento del comportamiento de los clientes y enviar mensajes personalizados manualmente.
- Segmente manualmente su base de clientes y envíe correos electrónicos individuales a cada grupo.

In [ ]:
q4_data = respuestas["q4"]
crear_quiz(q4_data["opciones"], q4_data["correcta"], q4_data["explicacion"])

### Pregunta 5

Una empresa de comercio electrónico está desarrollando una nueva plataforma en línea y quiere garantizar una experiencia de pago fluida y segura. ¿Cuáles son las ventajas de integrar su plataforma con una pasarela de pago de terceros?

- Elimina el riesgo de transacciones fraudulentas y filtraciones de datos
- Proporciona acceso a los datos de pago de los clientes para campañas específicas de publicidad y marketing
- Permite a la empresa centrarse en las funciones principales de la plataforma, como la exposición de productos y la experiencia del usuario
- Reduce la necesidad de crear un sistema de procesamiento de pagos desde cero, ahorrando tiempo y recursos de desarrollo

In [ ]:
q5_data = respuestas["q5"]
crear_quiz_multiple(q5_data["opciones"], q5_data["correctas"], q5_data["explicacion"])

### Pregunta 6

Un músico quiere promocionar su próxima gira de conciertos en redes sociales y automatizar el proceso de compartir las fechas y los enlaces a las entradas en diferentes plataformas. ¿Qué bibliotecas de Python serían más útiles?

- Instabot para automatizar las interacciones en Instagram
- Tweepy para compartir actualizaciones en Twitter
- Beautiful Soup para el raspado web de datos de Redes sociales
- Solicitudes (requests) para realizar peticiones HTTP a cualquier API de Redes sociales

In [ ]:
q6_data = respuestas["q6"]
crear_quiz_multiple(q6_data["opciones"], q6_data["correctas"], q6_data["explicacion"])

### Pregunta 7

Estás trabajando en un proyecto que requiere copias de seguridad diarias de tus datos en un servicio en la nube, y quieres automatizarlo para evitar olvidos y garantizar la coherencia. ¿Cuáles de los siguientes beneficios de las tareas programadas se aplican a este escenario?

- Eficiencia
- Seguridad mejorada
- Coherencia
- Automatización

In [ ]:
q7_data = respuestas["q7"]
crear_quiz_multiple(q7_data["opciones"], q7_data["correctas"], q7_data["explicacion"])

### Pregunta 8

Tienes una tarea cron que ejecuta un script para generar informes cada hora, pero observas que los informes no se generan. ¿Cuál de las siguientes razones podría ser la causa del fallo de la tarea cron?

- La ruta del script especificada en la entrada crontab es incorrecta.
- El demonio cron no se está ejecutando en su sistema.
- El archivo crontab tiene propiedad o permisos incorrectos.
- El reloj del sistema no está sincronizado con un servidor horario de red.

In [ ]:
q8_data = respuestas["q8"]
crear_quiz(q8_data["opciones"], q8_data["correcta"], q8_data["explicacion"])

### Pregunta 9

Estás automatizando una tarea de análisis de datos que debe ejecutarse cada 10 minutos. Sin embargo, consume muchos recursos y no quieres que interfiera con otros procesos del sistema. ¿Qué enfoque es el más adecuado?

- Explore marcos de programación alternativos, como Celery o APScheduler, que ofrecen funciones más avanzadas para la gestión de recursos y la priorización de tareas.
- Optimice el script de análisis de datos para reducir su consumo de recursos antes de programarlo con el módulo schedule.
- Ejecute manualmente la tarea cada 10 minutos durante las horas de menor actividad para minimizar el impacto en otros procesos.
- Utilice el módulo schedule para ejecutar la tarea cada 10 minutos, confiando en las capacidades de gestión de recursos integradas en el módulo.

In [ ]:
q9_data = respuestas["q9"]
crear_quiz(q9_data["opciones"], q9_data["correcta"], q9_data["explicacion"])

### Pregunta 10

Un equipo de desarrolladores de Python necesita programar tareas simples y recurrentes dentro de sus scripts, con un método fácil de aprender, compatible con varias plataformas y que permita mantener la lógica dentro del propio código Python. ¿Qué método se adaptaría mejor?

- Cron
- Funciones Azure
- El módulo schedule
- Programador de tareas

In [ ]:
q10_data = respuestas["q10"]
crear_quiz(q10_data["opciones"], q10_data["correcta"], q10_data["explicacion"])

## ¡Fin del examen!

Con esto cerraste el módulo 3: APIs, servicios de terceros y programación de tareas. Las tres
piezas juntas son el mismo concepto: **conectar un script con algo que ya sabe hacer el
trabajo** y dejarlo que corra solo.